## YOLO目标检测实验

本实验旨在快速上手YOLO（You Only Look Once）目标检测模型，涵盖环境搭建、模型推理（目标检测与姿态估计）以及迁移训练等核心环节。

### 0. 环境搭建

Ultralytics提供了YOLO的官方实现。建议使用清华镜像源加速下载：
```bash
pip install ultralytics -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple 
```

### 1. YOLO推理

Ultralytics 框架内置了在 COCO 数据集上预训练好的模型，可以直接识别 80 种常见的物体（如人、车、猫、狗等）。

#### 1.1 目标检测（Object Detection）

**命令行（处理单张图片）**

```bash
yolo predict model=yolo26n.pt source=bus.jpg show
```

**命令行（实时摄像头检测）**
```bash
yolo predict model=yolo26n.pt source=0 show
```

⚠️ 注意：使用命令行运行时，请确保终端的当前工作路径处于本代码所在的文件夹，否则可能会报找不到文件的错误。

**Python代码（图片/摄像头）**

使用Python API可以更方便地获取和处理检测到的数据（如坐标、置信度等）。

In [ ]:
from PIL import Image
from ultralytics import YOLO

# 加载预训练模型（nano版本，轻量快速）
model = YOLO("yolo26n.pt")

# 方式1：推理本地图片
im1 = Image.open("bus.jpg")
results = model.predict(source=im1, save=True)  # save=True保存标注结果

# 方式2：推理摄像头（取消注释即可使用）
# results = model.predict(source="0", save=True)

# 解析检测结果
for result in results:
    # 边界框坐标（多种格式）
    print("xyxy (像素坐标):", result.boxes.xyxy)   # (N, 4) 左上角+右下角
    print("xywh (像素坐标):", result.boxes.xywh)   # (N, 4) 中心点+宽高
    print("xyxyn (归一化):", result.boxes.xyxyn)   # (N, 4) 值范围[0,1]
    print("xywhn (归一化):", result.boxes.xywhn)   # (N, 4)
    
    # 置信度与类别
    print("置信度:", result.boxes.conf)   # (N, 1)
    print("类别ID:", result.boxes.cls)    # (N, 1)
    
# 可视化结果（在Jupyter中显示）
# from IPython.display import Image as IPImage
# IPImage(filename='runs/detect/predict/bus.jpg')  # 根据实际保存路径调整

#### 1.2 关节点检测（Pose Estimation）

除了检测物体在哪里，YOLO还能识别出人体关键点（如肩膀、手肘、膝盖等）。我们只需更换一个带 -pose 后缀的模型即可。

命令行（摄像头）
```bash
yolo predict model=yolo26n-pose.pt source=0 show
```


Python代码

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n-pose.pt")  # load an official model

# Predict with the model
results = model("bus.jpg")  # predict on an image

# Access the results
for result in results:
    xy = result.keypoints.xy  # x and y coordinates
    xyn = result.keypoints.xyn  # normalized
    kpts = result.keypoints.data  # x, y, visibility (if available)
    print(xy, xyn, kpts)

### 2. 迁移训练

在实际业务中，我们通常需要识别 COCO 数据集之外的特定物体（比如工业零件、特定商品的包装等）。这就需要使用我们自己的数据集对模型进行微调（迁移训练）。

#### 2.1 构造数据集

数据集结构要求

```
my_dataset/
├── images/
│   ├── train/
│   │   ├── img1.jpg
│   │   └── ...
│   └── val/
│       ├── img1.jpg
│       └── ...
├── labels/
│   ├── train/
│   │   ├── img1.txt
│   │   └── ...
│   └── val/
│       ├── img1.txt
│       └── ...
└── data.yaml
```

data.yaml示例

```yaml
path: ../my_dataset  # 数据集根目录
train: images/train  # 训练图片目录
val: images/val      # 验证图片目录

nc: 2                # 类别数量
names: ['cat', 'dog'] # 类别名称
```

#### 2.2 开始迁移训练

这里以一个 **饮料瓶数据集（Drink_284）** 为例进行训练。

- data: 指定你的 yaml 配置文件路径。
- pretrained: 指定基础模型，我们在此基础上微调。
- epochs: 训练轮数（设为1仅供跑通代码测试，实际训练建议设为 50-100+）。
- batch: 批次大小（如果显存/内存不足，可以调小为 8 或 4）。
- imgsz: 图像输入尺寸，默认 640。

```bash
yolo detect train data=data.yaml pretrained=yolov26n.pt epochs=1 batch=16
```

训练完成后，模型会自动将验证集的评估指标（mAP等）以及最好的权重文件 `best.pt` 保存到 `runs/detect/train/weights/` 目录下。

#### 2.3 检测训练效果

使用刚刚训练得到的“最优权重（best.pt）”，在我们的验证集图片上进行测试，看看模型是否学会了识别目标物体

```bash
yolo predict model=runs/detect/train/weights/best.pt source=./dataset/images/val
```